# Stratified Distributed SGD for Matrix Factorization — semantic block partitioning on ML-1M

Simulation accompanying the report on **Gemulla, Haas, Nijkamp & Sismanis, *Large-Scale Matrix
Factorization with Distributed Stochastic Gradient Descent*, KDD 2011**.

**Question.** DSGD builds its $d\times d$ block grid from *random* row/column permutations. What
happens when the grid is *semantic* instead — movies clustered by genre or release year, users
grouped by demographic age or by history similarity — as in a naturally sharded deployment? And
does that choice propagate to the **SPINRec explanation fidelity** (CDCG@k, POS@10) of the model?

**What this notebook does**
1. Builds a laptop-sized ML-1M subsample (leave-one-out by timestamp).
2. Implements the recommender used in the thesis pipeline (`refiend_metrics/models.py`) as an
   explicit bilinear factorization, and trains it with a faithful DSGD simulation.
3. Verifies Gemulla's Theorem 2 numerically on classic MF, and measures where it breaks for the
   history-encoder model.
4. Evaluates SPINRec deletion fidelity per blocking scheme, with paired statistics.
5. Runs a martingale/drift diagnostic with an Azuma envelope.
6. Writes all figures and LaTeX tables for the report.

Set `FAST = False` for the full configuration. Measured runtime on an M4 MacBook Pro: **5.6 min**
(FAST); roughly 20 min for the full configuration.

In [1]:

import json, time, sys
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans
from scipy.stats import wilcoxon, spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ML-1M lives in different places on different machines; take the first complete copy
_DATA_CANDIDATES = [
    Path('/Users/yagelalfasi/Downloads/BEYONDTOP1-main/processed_data/ML1M/data files'),
    Path('/Users/yagelalfasi/Desktop/LXR_/processed_data/ML1M'),
    Path('/Users/yagelalfasi/Documents/main/RobustnessCheck/processed_data/ML1M'),
]
_NEEDED = ('ratings.dat', 'movies.dat', 'users.dat')
DATA = next((p for p in _DATA_CANDIDATES if all((p / f).exists() for f in _NEEDED)), None)
if DATA is None:
    raise FileNotFoundError('ML-1M not found; add its directory to _DATA_CANDIDATES. '
                            'Needs ratings.dat, movies.dat, users.dat')
print('data:', DATA)
FIG  = Path('/Users/yagelalfasi/Documents/Agents/figures')
FIG.mkdir(exist_ok=True)

FAST = False                               # True = ~6 min smoke run; False = the reported configuration
CFG = dict(k=32, lr=1.0, n_neg=4, batch=256, d=4, epochs=10 if FAST else 20)
SEEDS         = [0, 1] if FAST else [0, 1, 2, 3, 4]
N_EXPL_USERS  = 120 if FAST else 800       # users explained by SPINRec
R_BASELINES   = 3 if FAST else 5           # stochastic baselines per user
K_MASK        = 10                         # masking steps
ARMS = ['RAND', 'FIXED', 'GENRE', 'GENRE-BAL', 'YEAR', 'DEMOG', 'USER-HIST']
D_GRID        = [2, 4, 8]                  # how the violation scales with the block count
D_SWEEP_ARMS  = ['RAND', 'FIXED', 'GENRE', 'USER-HIST']   # DEMOG is defined for d=4 only
D_SEEDS       = [0] if FAST else [0, 1, 2]
t_start = time.time()
print('config:', CFG, '| seeds', SEEDS)

data: /Users/yagelalfasi/Downloads/BEYONDTOP1-main/processed_data/ML1M/data files
config: {'k': 32, 'lr': 1.0, 'n_neg': 4, 'batch': 256, 'd': 4, 'epochs': 20} | seeds [0, 1, 2, 3, 4]


## 1. Data

ML-1M, binarised at `rating >= 4` (the convention used in the thesis pipeline). We keep the 1,500
most-rated movies and a random sample of 2,000 users with at least 10 positives among them, which
leaves ~170K training interactions — small enough that a full sweep of 6 schemes x seeds runs in
minutes, large enough that the block structure is meaningful.

Split: leave-one-out by timestamp (last positive -> test, previous -> validation).

In [2]:
def load_ml1m(n_items=1500, n_users=2000, pos_threshold=4, min_pos=10, seed=0):
    r = pd.read_csv(DATA / 'ratings.dat', sep='::', engine='python',
                    names=['user', 'item', 'rating', 'ts'], encoding='latin-1')
    pos = r[r.rating >= pos_threshold].copy()
    top_items = pos.item.value_counts().head(n_items).index
    pos = pos[pos.item.isin(top_items)]
    cnt = pos.user.value_counts()
    ok_users = cnt[cnt >= min_pos].index.to_numpy()
    rng = np.random.default_rng(seed)
    keep = rng.choice(ok_users, size=min(n_users, len(ok_users)), replace=False)
    pos = pos[pos.user.isin(keep)]

    uids = np.sort(pos.user.unique()); iids = np.sort(pos.item.unique())
    umap = {u: k for k, u in enumerate(uids)}; imap = {i: k for k, i in enumerate(iids)}
    pos['u'] = pos.user.map(umap); pos['i'] = pos.item.map(imap)
    return pos, uids, iids


def loo_split(pos, U, I):
    """Leave-one-out by timestamp: last positive -> test, previous -> validation."""
    pos = pos.sort_values(['u', 'ts'], kind='mergesort')
    last = pos.groupby('u').tail(1)
    rest = pos.drop(last.index)
    val = rest.groupby('u').tail(1)
    train = rest.drop(val.index)

    X = np.zeros((U, I), dtype=np.float32)
    X[train.u.to_numpy(), train.i.to_numpy()] = 1.0
    test_item = np.full(U, -1, dtype=np.int64); val_item = np.full(U, -1, dtype=np.int64)
    test_item[last.u.to_numpy()] = last.i.to_numpy()
    val_item[val.u.to_numpy()] = val.i.to_numpy()
    return X, train, test_item, val_item

def load_side_info(iids, uids):
    m = pd.read_csv(DATA / 'movies.dat', sep='::', engine='python',
                    names=['item', 'title', 'genres'], encoding='latin-1').set_index('item')
    us = pd.read_csv(DATA / 'users.dat', sep='::', engine='python',
                     names=['user', 'gender', 'age', 'occ', 'zip'], encoding='latin-1').set_index('user')
    GEN = ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary',
           'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance',
           'Sci-Fi', 'Thriller', 'War', 'Western']
    gidx = {g: k for k, g in enumerate(GEN)}
    G = np.zeros((len(iids), len(GEN)), dtype=np.float32)
    year = np.zeros(len(iids), dtype=np.int32)
    titles = []
    for k, it in enumerate(iids):
        row = m.loc[it]
        for g in str(row.genres).split('|'):
            if g in gidx:
                G[k, gidx[g]] = 1.0
        t = str(row.title); titles.append(t)
        try:
            year[k] = int(t.strip()[-5:-1])
        except ValueError:
            year[k] = 1990
    age = us.loc[uids, 'age'].to_numpy()
    return G, year, age, GEN, titles

In [3]:

pos, uids, iids = load_ml1m()
U, I = len(uids), len(iids)
X, train, test_item, val_item = loo_split(pos, U, I)
pu, pi = train.u.to_numpy(), train.i.to_numpy()
G, year, age, GENRES, titles = load_side_info(iids, uids)
item_nnz = np.bincount(pi, minlength=I).astype(float)

print(f'users {U}  items {I}  train interactions {int(X.sum())}  density {X.sum()/(U*I):.3f}')
print(f'history length: mean {X.sum(1).mean():.1f}  median {np.median(X.sum(1)):.0f}  '
      f'min {int(X.sum(1).min())}  max {int(X.sum(1).max())}')
print('age brackets:', dict(zip(*np.unique(age, return_counts=True))))
print('mean genres per movie: %.2f   release years: %d-%d' % (G.sum(1).mean(), year.min(), year.max()))

users 2000  items 1500  train interactions 168980  density 0.056
history length: mean 84.5  median 52  min 8  max 819
age brackets: {np.int64(1): np.int64(79), np.int64(18): np.int64(355), np.int64(25): np.int64(707), np.int64(35): np.int64(380), np.int64(45): np.int64(175), np.int64(50): np.int64(179), np.int64(56): np.int64(125)}
mean genres per movie: 1.93   release years: 1922-2000


## 2. The model

The recommender in the thesis pipeline (`refiend_metrics/models.py`, class `MLP`) scores a
(user, item) pair as

$$ s(u,j) \;=\; \sigma\!\big( (A^\top x_u)\cdot(B^\top e_j) \big), $$

where $x_u\in\{0,1\}^{I}$ is the user's **history vector** and items are one-hot
(`data_processing.py` sets `items_array = np.eye(num_items)`). So the item factor
$h_j = B_{j,:}$ is private to item $j$, but the user factor

$$ w_u \;=\; \sum_{i \in \mathrm{hist}(u)} A_{i,:} $$

is assembled from **item-indexed rows of $A$ that different users share**. This is the crux of the
whole study: Gemulla's interchangeability (Def. 1) assumes the row factor is private to the row, so
DSGD is *exact* for classic MF and only *approximate* here.

Two deliberate deviations from the thesis module, both required for a clean experiment:

* **No biases.** `nn.Linear(..., bias=True)` adds $b_a, b_b$ which every training example touches, so
  *all* pairs of examples would conflict — the interchangeability violation would then be trivially
  true for a reason unrelated to history overlap.
* **In-block negative sampling.** A node only holds its own item shard, so it can only draw negatives
  from that shard. This keeps the $B$ rows disjoint across a stratum. It has a real side effect
  (genre-homogeneous shards yield genre-matched *hard* negatives), which we quantify in an ablation
  rather than leave silent.

In [4]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def sample_negatives(rng, X, users, pool, n_neg):
    """Uniform negatives from `pool` (item ids), rejecting observed items (up to 3 tries)."""
    n = len(users)
    negs = rng.choice(pool, size=(n, n_neg))
    for _ in range(3):
        bad = X[users[:, None], negs] > 0
        if not bad.any():
            break
        negs[bad] = rng.choice(pool, size=int(bad.sum()))
    return negs


def batch_grads(X, A, B, users, items, labels):
    """Gradients of mean BCE over a batch of (user, item, label) triples."""
    Xb = X[users]                      # (n, I)
    Wu = Xb @ A                        # (n, k)
    Hj = B[items]                      # (n, k)
    e = (sigmoid(np.einsum('nk,nk->n', Wu, Hj)) - labels) / len(users)
    Gk = e[:, None] * Hj               # (n, k) -> dL/dWu
    gA = Xb.T @ Gk                     # (I, k)
    gB = np.zeros_like(B)
    np.add.at(gB, items, e[:, None] * Wu)
    return gA, gB


def sgd_pass(X, A, B, pos_u, pos_i, pool, rng, lr, n_neg, batch, max_batches=None):
    """One local SGD pass over the given positives. Mutates A, B in place."""
    order = rng.permutation(len(pos_u))
    starts = range(0, len(order), batch)
    if max_batches is not None:
        starts = list(starts)[:max_batches]
    for s in starts:
        sel = order[s:s + batch]
        u, i = pos_u[sel], pos_i[sel]
        negs = sample_negatives(rng, X, u, pool, n_neg)
        users = np.repeat(u, 1 + n_neg)
        items = np.concatenate([i[:, None], negs], axis=1).ravel()
        labels = np.tile(np.concatenate([[1.0], np.zeros(n_neg)]), len(u)).astype(np.float32)
        gA, gB = batch_grads(X, A, B, users, items, labels)
        A -= lr * gA
        B -= lr * gB
    return A, B

In [5]:

def init_params(I, k, seed, scale=0.05):
    r = np.random.default_rng(1000 + seed)
    return ((r.standard_normal((I, k)) * scale).astype(np.float32),
            (r.standard_normal((I, k)) * scale).astype(np.float32))


def evaluate(A, B, ep=None):
    '''HR@10 and MPR against held-out positives, training history excluded.'''
    S = (X @ A) @ B.T
    S[X > 0] = -1e9
    out = {}
    for name, tgt in (('val', val_item), ('test', test_item)):
        ok = tgt >= 0
        r = (S[ok] > S[ok, tgt[ok]][:, None]).sum(1) + 1
        out[f'{name}_hr10'] = float((r <= 10).mean())
        out[f'{name}_mpr'] = float(r.mean() / I)
    return out

## 3. Block partitioning schemes

The grid is $d\times d$; a **stratum** is a $d$-monomial (one block per row group and per column
group), and an epoch cycles through $d$ strata that together cover the matrix once.

| scheme | user groups | item groups | mechanism it moves |
|---|---|---|---|
| `RAND` | random, re-drawn each epoch | random, re-drawn each epoch | paper default |
| `FIXED` | random, fixed | random, fixed | isolates re-permutation from partition structure |
| `GENRE` | random | k-means on the 18-dim genre multi-hot | load imbalance, per-stratum gradient bias |
| `GENRE-BAL` | random | genre clusters rebalanced to equal nnz | separates load from semantics |
| `YEAR` | random | release-year quartiles | as `GENRE` |
| `DEMOG` | ML-1M age brackets | random | shared-parameter collisions |
| `USER-HIST` | k-means on user history vectors | random | collisions, strong dose |

**Why user-side and item-side schemes are not symmetric.** A stratum contains one block per row
group, so *every stratum activates all $d$ user groups at once*; and the rows of $A$ a block touches
are $\bigcup_{u \in \text{row group}} \mathrm{hist}(u)$ — the forward pass consumes the user's whole
history, not only the block's items. Collisions on the shared $A$ are therefore governed by the
**user** partition alone; item-side schemes act on load balance and stratum bias instead.

In [6]:
def rebalance(labels, weight, d, tol=0.05, max_moves=100000):
    """Greedily move the lightest members from heavy groups to light ones until
    every group's total `weight` is within `tol` of the mean."""
    labels = labels.copy()
    for _ in range(max_moves):
        tot = np.array([weight[labels == g].sum() for g in range(d)])
        mean = tot.mean()
        if tot.max() <= mean * (1 + tol) and tot.min() >= mean * (1 - tol):
            break
        hi, lo = int(tot.argmax()), int(tot.argmin())
        cand = np.where(labels == hi)[0]
        if len(cand) <= 1:
            break
        labels[cand[np.argmin(weight[cand])]] = lo
    return labels


def enforce_min_size(labels, d, min_frac=0.10, feats=None):
    """Guard against degenerate k-means clusters (empty strata -> NaNs)."""
    labels = labels.copy(); n = len(labels); need = int(min_frac * n / 1)
    for _ in range(1000):
        sizes = np.bincount(labels, minlength=d)
        if sizes.min() >= need:
            break
        lo, hi = int(sizes.argmin()), int(sizes.argmax())
        cand = np.where(labels == hi)[0]
        if feats is not None:
            c_lo = feats[labels == lo].mean(0) if sizes[lo] else feats[cand].mean(0)
            cand = cand[np.argsort(((feats[cand] - c_lo) ** 2).sum(1))]
        labels[cand[0]] = lo
    return labels


def build_strata(d):
    """Cyclic Latin square: d strata, each a d-monomial covering every row/col group once."""
    return [[(r, (r + t) % d) for r in range(d)] for t in range(d)]

def gini(x):
    """Gini coefficient of a non-negative vector (0 = perfectly balanced blocks)."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))

In [7]:
def make_partition(scheme, X, G, year, age, d, rng, item_nnz):
    """Return (row_labels, col_labels, reshuffle_each_epoch)."""
    U, I = X.shape
    rand_rows = rng.integers(0, d, U)
    rand_cols = rng.integers(0, d, I)
    if scheme == 'RAND':
        return rand_rows, rand_cols, True
    if scheme == 'FIXED':
        # random partition held fixed across epochs: isolates re-permutation from the
        # partition's structure, since RAND is the only scheme that re-draws each epoch
        return rand_rows, rand_cols, False
    if scheme == 'GENRE':
        lab = KMeans(d, n_init=10, random_state=0).fit_predict(G)
        return rand_rows, enforce_min_size(lab, d, min(0.10, 0.5 / d), G), False
    if scheme == 'GENRE-BAL':
        lab = enforce_min_size(KMeans(d, n_init=10, random_state=0).fit_predict(G), d, min(0.10, 0.5 / d), G)
        return rand_rows, rebalance(lab, item_nnz, d), False
    if scheme == 'YEAR':
        q = np.quantile(year, np.linspace(0, 1, d + 1)[1:-1])
        return rand_rows, np.digitize(year, q), False
    if scheme == 'DEMOG':
        # ML-1M age brackets merged into 4 groups of comparable size
        if d != 4:
            raise ValueError('DEMOG is defined for d=4 only (7 ML-1M age brackets -> 4 groups)')
        mapping = {1: 0, 18: 0, 25: 1, 35: 2, 45: 3, 50: 3, 56: 3}
        return np.array([mapping[a] for a in age]), rand_cols, False
    if scheme == 'USER-HIST':
        lab = KMeans(d, n_init=10, random_state=0).fit_predict(X)
        return enforce_min_size(lab, d, min(0.10, 0.5 / d), X), rand_cols, False
    raise ValueError(scheme)


def block_index(pos_u, pos_i, row_lab, col_lab, d):
    """idx[(r,c)] -> positions of that block's positives."""
    key = row_lab[pos_u] * d + col_lab[pos_i]
    order = np.argsort(key, kind='stable')
    ks = key[order]
    bounds = np.searchsorted(ks, np.arange(d * d + 1))
    return {(b // d, b % d): order[bounds[b]:bounds[b + 1]] for b in range(d * d)}

## 4. The DSGD engine

One subepoch: all $d$ blocks of the current stratum step **from the same starting parameters**, then
their deltas are summed. For disjoint parameters that is exactly the paper's parallel execution; for
the shared $A$ it is the natural approximation.

Diagnostics recorded per subepoch:
* **contested gradient mass** — for each row of $A$, the total delta mass from all blocks minus the
  largest single block's share, normalised. (The naive "fraction of rows touched by $\ge2$ blocks"
  saturates near 1 for every scheme and discriminates nothing.)
* **straggler ratio** $\max/\mathrm{mean}$ of block sizes — the makespan model for one subepoch.
* the projected **gradient-noise increment** for the martingale diagnostic (Section 7).

In [8]:
def run_sequential(X, pos_u, pos_i, cfg, seed, eval_fn):
    A, B = init_params(X.shape[1], cfg['k'], seed)
    pool = np.arange(X.shape[1])
    hist = []
    for ep in range(cfg['epochs']):
        rng = np.random.default_rng([seed, ep, 999])
        sgd_pass(X, A, B, pos_u, pos_i, pool, rng, cfg['lr'], cfg['n_neg'], cfg['batch'])
        hist.append(eval_fn(A, B, ep))
    return A, B, hist


def run_dsgd(X, pos_u, pos_i, row_lab, col_lab, cfg, seed, eval_fn, G=None, year=None,
             age=None, scheme=None, reshuffle=False, diag_epochs=2, weighted=False,
             martingale=False, global_negatives=False):
    """Block-parallel DSGD: all d blocks of a stratum step from shared parameters,
    deltas are summed (exact for disjoint parameters, approximate otherwise)."""
    U, I = X.shape
    d = cfg['d']
    A, B = init_params(I, cfg['k'], seed)
    strata = build_strata(d)
    all_items = np.arange(I)
    hist, diag = [], {'t2_gap': [], 'overlap': [], 'straggler': [], 'block_nnz': [],
                      'mart_inc': [], 'stratum_nnz': []}
    v = np.random.default_rng(7).standard_normal(I * cfg['k']).astype(np.float32)
    v /= np.linalg.norm(v)

    for ep in range(cfg['epochs']):
        if reshuffle:
            r2 = np.random.default_rng([seed, ep, 5])
            row_lab = r2.integers(0, d, U); col_lab = r2.integers(0, d, I)
        bidx = block_index(pos_u, pos_i, row_lab, col_lab, d)
        col_items = [np.where(col_lab == c)[0] for c in range(d)]

        for t, stratum in enumerate(strata):
            A0, B0 = A.copy(), B.copy()
            s_nnz = sum(len(bidx[b]) for b in stratum)
            lr = cfg['lr'] * (s_nnz / (len(pos_u) / d)) if weighted else cfg['lr']
            if martingale and ep < cfg['epochs']:
                gA_full, gB_full = full_gradient(X, A0, B0, pos_u, pos_i, cfg['n_neg'])

            dA, dB, nb = [], [], 0
            for (r, c) in stratum:
                idx = bidx[(r, c)]
                if len(idx) == 0:
                    dA.append(0.0); dB.append(0.0); continue
                Al, Bl = A0.copy(), B0.copy()
                rng = np.random.default_rng([seed, ep, t, r])
                pool = all_items if global_negatives else col_items[c]
                sgd_pass(X, Al, Bl, pos_u[idx], pos_i[idx], pool, rng, lr,
                         cfg['n_neg'], cfg['batch'])
                dA.append(Al - A0); dB.append(Bl - B0)
                nb += int(np.ceil(len(idx) / cfg['batch']))

            A = A0 + sum(dA); B = B0 + sum(dB)

            nnz = [len(bidx[b]) for b in stratum]
            diag['block_nnz'].append(nnz); diag['stratum_nnz'].append(s_nnz)
            diag['straggler'].append(max(nnz) / max(np.mean(nnz), 1e-9))
            mass = np.array([np.linalg.norm(x, axis=1) if np.ndim(x) else np.zeros(I)
                             for x in dA])                       # (d, I) row-norms
            tot = mass.sum(0)
            diag['overlap'].append(float((tot - mass.max(0)).sum() / max(tot.sum(), 1e-12)))

            if martingale:
                inc = (np.concatenate([(A - A0).ravel(), ]) / -lr) - nb * gA_full.ravel()
                diag['mart_inc'].append(float(inc @ v))

            if ep < diag_epochs:                                  # Theorem-2 gap
                As, Bs = A0.copy(), B0.copy()
                for (r, c) in stratum:
                    idx = bidx[(r, c)]
                    if len(idx) == 0:
                        continue
                    rng = np.random.default_rng([seed, ep, t, r])
                    pool = all_items if global_negatives else col_items[c]
                    sgd_pass(X, As, Bs, pos_u[idx], pos_i[idx], pool, rng, lr,
                             cfg['n_neg'], cfg['batch'])
                num = np.linalg.norm(A - As) + np.linalg.norm(B - Bs)
                den = np.linalg.norm(A - A0) + np.linalg.norm(B - B0)
                diag['t2_gap'].append(float(num / max(den, 1e-12)))

        hist.append(eval_fn(A, B, ep))
    return A, B, hist, diag

## 5. Theorem 2, numerically

Gemulla's Theorem 2 says training points that share neither a row nor a column are
*interchangeable*: running them in any order, including in parallel, gives exactly the same result.

**Control.** Classic free-embedding MF ($P\in\mathbb R^{U\times k}$, $Q\in\mathbb R^{I\times k}$) has
disjoint block parameters, so parallel-merge must equal sequential application to machine precision.

**Test.** For the history-encoder model we measure the relative deviation
$\|\theta_{\rm par}-\theta_{\rm seq}\|/\|\theta_{\rm par}-\theta_0\|$ *as a function of how many local
mini-batch steps each block takes*. Measuring it after a whole local pass would confound the
collision effect with how far each block travels; the curve isolates it.

In [9]:
def theorem2_curve(X, pos_u, pos_i, row_lab, col_lab, cfg, A0, B0, steps_grid, seed=0,
                   global_negatives=False):
    """Relative deviation ||theta_par - theta_seq|| / ||theta_par - theta_0|| as a function
    of the number of local mini-batch steps each block takes inside one subepoch.

    Theorem 2 says this is identically 0 when block parameters are disjoint; for the
    bilinear (history-encoder) model it grows with the number of local steps, and the
    growth rate is the quantity the partition scheme controls.
    """
    d = cfg['d']
    bidx = block_index(pos_u, pos_i, row_lab, col_lab, d)
    col_items = [np.where(col_lab == c)[0] for c in range(d)]
    all_items = np.arange(X.shape[1])
    stratum = [(r, r) for r in range(d)]
    out = []
    for ns in steps_grid:
        dA = dB = 0
        for (r, c) in stratum:
            idx = bidx[(r, c)]
            if len(idx) == 0:
                continue
            Al, Bl = A0.copy(), B0.copy()
            sgd_pass(X, Al, Bl, pos_u[idx], pos_i[idx],
                     all_items if global_negatives else col_items[c],
                     np.random.default_rng([seed, r]), cfg['lr'], cfg['n_neg'],
                     cfg['batch'], max_batches=ns)
            dA = dA + (Al - A0); dB = dB + (Bl - B0)
        A_par, B_par = A0 + dA, B0 + dB

        As, Bs = A0.copy(), B0.copy()
        for (r, c) in stratum:
            idx = bidx[(r, c)]
            if len(idx) == 0:
                continue
            sgd_pass(X, As, Bs, pos_u[idx], pos_i[idx],
                     all_items if global_negatives else col_items[c],
                     np.random.default_rng([seed, r]), cfg['lr'], cfg['n_neg'],
                     cfg['batch'], max_batches=ns)
        num = np.sqrt(np.sum((A_par - As) ** 2) + np.sum((B_par - Bs) ** 2))
        den = np.sqrt(np.sum((A_par - A0) ** 2) + np.sum((B_par - B0) ** 2))
        out.append(float(num / max(den, 1e-12)))
    return out

def theorem2_curve_classic(X, pos_u, pos_i, d, steps_grid, k=8, seed=0, batch=256,
                           lr=0.5, n_neg=4):
    """Same curve for classic free-embedding MF: must stay at machine zero."""
    return [run_classic_mf_control(X, pos_u, pos_i, d, k=k, seed=seed, batch=batch,
                                   lr=lr, n_neg=n_neg, max_batches=ns, relative=True)
            for ns in steps_grid]

def run_classic_mf_control(X, pos_u, pos_i, d, k=8, seed=0, batch=256, lr=0.5, n_neg=4,
                           max_batches=None, relative=False):
    """Free per-user embeddings: block parameters are disjoint, so parallel-merge
    must equal sequential application within a stratum (Gemulla Thm. 2)."""
    U, I = X.shape
    rng0 = np.random.default_rng(seed)
    P = rng0.standard_normal((U, k)) * 0.05          # float64 on purpose
    Q = rng0.standard_normal((I, k)) * 0.05
    row_lab = rng0.integers(0, d, U); col_lab = rng0.integers(0, d, I)
    bidx = block_index(pos_u, pos_i, row_lab, col_lab, d)
    col_items = [np.where(col_lab == c)[0] for c in range(d)]
    stratum = [(r, r) for r in range(d)]

    def local(P0, Q0, idx, c, seed_b):
        Pl, Ql = P0.copy(), Q0.copy()
        rng = np.random.default_rng(seed_b)
        order = rng.permutation(len(idx))
        starts = range(0, len(order), batch)
        if max_batches is not None:
            starts = list(starts)[:max_batches]
        for s in starts:
            sel = idx[order[s:s + batch]]
            u, i = pos_u[sel], pos_i[sel]
            negs = rng.choice(col_items[c], size=(len(u), n_neg))
            users = np.repeat(u, 1 + n_neg)
            items = np.concatenate([i[:, None], negs], axis=1).ravel()
            y = np.tile(np.concatenate([[1.0], np.zeros(n_neg)]), len(u))
            e = (sigmoid((Pl[users] * Ql[items]).sum(1)) - y) / len(users)
            gP = np.zeros_like(Pl); gQ = np.zeros_like(Ql)
            np.add.at(gP, users, e[:, None] * Ql[items])
            np.add.at(gQ, items, e[:, None] * Pl[users])
            Pl -= lr * gP; Ql -= lr * gQ
        return Pl, Ql

    P0, Q0 = P.copy(), Q.copy()
    dP = dQ = 0
    for (r, c) in stratum:
        Pl, Ql = local(P0, Q0, bidx[(r, c)], c, [seed, r])
        dP = dP + (Pl - P0); dQ = dQ + (Ql - Q0)
    P_par, Q_par = P0 + dP, Q0 + dQ

    Ps, Qs = P0.copy(), Q0.copy()
    for (r, c) in stratum:
        Ps, Qs = local(Ps, Qs, bidx[(r, c)], c, [seed, r])
    if relative:
        num = np.sqrt(np.sum((P_par - Ps) ** 2) + np.sum((Q_par - Qs) ** 2))
        den = np.sqrt(np.sum((P_par - P0) ** 2) + np.sum((Q_par - Q0) ** 2))
        return float(num / max(den, 1e-12))
    return float(np.abs(P_par - Ps).max() + np.abs(Q_par - Qs).max())

In [10]:

gaps = [run_classic_mf_control(X, pu, pi, d=CFG['d'], seed=s) for s in range(3)]
print('Theorem-2 control on classic MF: max |parallel - sequential| = %.2e  (machine zero)'
      % np.max(gaps))

STEPS_GRID = [1, 2, 5, 10, 20, 50, 100]
A_ck, B_ck = init_params(I, CFG['k'], 0)
for _ in range(2):                      # measure at a trained point, not at initialisation
    sgd_pass(X, A_ck, B_ck, pu, pi, np.arange(I), np.random.default_rng(77),
             CFG['lr'], CFG['n_neg'], CFG['batch'])

t2curve = {'steps': STEPS_GRID,
           'CLASSIC-MF': theorem2_curve_classic(X, pu, pi, CFG['d'], STEPS_GRID)}
for arm in ARMS:
    rl, cl, _ = make_partition(arm, X, G, year, age, CFG['d'], np.random.default_rng(0), item_nnz)
    t2curve[arm] = theorem2_curve(X, pu, pi, rl, cl, CFG, A_ck, B_ck, STEPS_GRID)
    print('%-10s %s' % (arm, np.round(t2curve[arm], 4)))

Theorem-2 control on classic MF: max |parallel - sequential| = 8.13e-20  (machine zero)


RAND       [0.0167 0.0313 0.0679 0.0832 0.1089 0.1234 0.1234]


FIXED      [0.0167 0.0313 0.0679 0.0832 0.1089 0.1234 0.1234]


GENRE      [0.0203 0.0347 0.0644 0.0774 0.1026 0.1411 0.1435]


GENRE-BAL  [0.0368 0.061  0.118  0.1613 0.164  0.1621 0.1621]


YEAR       [0.0535 0.0916 0.121  0.1178 0.1299 0.151  0.151 ]


DEMOG      [0.0196 0.0476 0.0662 0.0687 0.1075 0.1294 0.1271]


USER-HIST  [0.0829 0.1284 0.2082 0.2526 0.3087 0.3477 0.323 ]


## 6. SPINRec explanation fidelity

SPINRec explains a recommendation with integrated gradients w.r.t. the user's history vector, using a
*stochastic* baseline (a randomly drawn training user), attribution
$x \odot \bar g \odot (x-b)$.

For this bilinear model the attribution is **closed-form**: since
$\partial s/\partial x = \sigma'(z)\,(A h_j)$ and $\sigma'$ is a positive scalar that factors out of
the ranking,

$$ \text{attr}_i \;=\; \Big(\tfrac1T\sum_t \sigma'(z_t)\Big)\, x_i\,(x_i-b_i)\,\big(A h_j\big)_i . $$

So the explanation ranking is a direct read-out of $A B^\top$ restricted to the user's history — which
is exactly the object the blocking scheme perturbs. This is ~100x faster than looping autograd, and
we cross-check it against torch below.

*(Note: the repo copy at `refiend_metrics/SPINRec_functions.py:190` contains a `break` that truncates
the integration to a single step; this notebook implements the untruncated definition.)*

Metrics, following the thesis pipeline: mask the top-$k$ attributed history items and re-rank —
**CDCG@k** $=1/\log_2(\text{rank}+1)$ with the target ranked among *all* items, and **POS@10**, the
probability the target survives in the top 10.

In [11]:
def spinrec_attribution(A, B, x, j, baselines, n_steps=20):
    """Integrated gradients w.r.t. the history vector with stochastic baselines.

    For this bilinear model dS/dx = sigma'(z) * (A @ h_j); the scalar sigma' factors
    out of the ranking, so the attribution is closed-form (verified against autograd).
    """
    q = A @ B[j]
    out = np.zeros_like(q)
    for b in baselines:
        delta = x - b
        ts = np.linspace(0, 1, n_steps + 1)[1:]
        z = ((b[None, :] + ts[:, None] * delta[None, :]) @ A) @ B[j]
        s = sigmoid(z)
        c = float(np.mean(s * (1 - s)))
        out += x * delta * c * q
    return out / len(baselines)


def spinrec_fidelity(A, B, X, users, baselines_per_user, K=10, n_steps=20):
    """CDCG@k and POS@10 curves under progressive masking of top-attributed items."""
    S_full = (X[users] @ A) @ B.T
    j_star = S_full.argmax(1)
    Xu = X[users].copy()
    Xu[np.arange(len(users)), j_star] = 0.0

    attrs = []
    for n, u in enumerate(users):
        attrs.append(spinrec_attribution(A, B, Xu[n], j_star[n], baselines_per_user[n], n_steps))
    attrs = np.array(attrs)

    cdcg = np.zeros((len(users), K)); pos10 = np.zeros((len(users), K))
    order = [np.where(Xu[n] > 0)[0][np.argsort(-attrs[n][Xu[n] > 0])] for n in range(len(users))]
    for k in range(1, K + 1):
        Xm = Xu.copy()
        for n in range(len(users)):
            Xm[n, order[n][:k]] = 0.0
        S = (Xm @ A) @ B.T
        tgt = S[np.arange(len(users)), j_star]
        rank = (S > tgt[:, None]).sum(1) + 1
        cdcg[:, k - 1] = 1.0 / np.log2(rank + 1)
        pos10[:, k - 1] = (rank <= 10)
    return cdcg, pos10, attrs, j_star

In [12]:

# cross-check the closed form against torch autograd
import torch
_A, _B = init_params(I, CFG['k'], 0)
_u = 3; _x = X[_u].copy(); _j = int(((_x @ _A) @ _B.T).argmax()); _x[_j] = 0
_b = X[17].astype(np.float32)
_At, _Bt = torch.tensor(_A), torch.tensor(_B)
_g = []
for _t in np.linspace(0, 1, 21)[1:].astype(np.float32):
    _xi = torch.tensor(_b + _t * (_x - _b), requires_grad=True)
    torch.sigmoid((_xi @ _At) @ _Bt[_j]).backward()
    _g.append(_xi.grad.numpy())
_ref = _x * (_x - _b) * np.mean(_g, 0)
_ours = spinrec_attribution(_A, _B, _x, _j, [_b], n_steps=20)
print('closed form vs autograd: max abs diff %.2e, correlation %.10f'
      % (np.abs(_ref - _ours).max(), np.corrcoef(_ref, _ours)[0, 1]))

closed form vs autograd: max abs diff 1.86e-09, correlation 1.0000000000


### Common random numbers

Every arm is evaluated on the **same** explained users with the **same** baseline draws and the same
parameter initialisation. Without this, the variance of the stochastic baseline alone exceeds the
effect we are trying to measure, and 2-3 unpaired seed means would show nothing.

In [13]:

hist_len = X.sum(1)
expl_users = np.random.default_rng(4242).choice(np.where(hist_len >= 20)[0],
                                                N_EXPL_USERS, replace=False)
_brng = np.random.default_rng(12345)
baselines = [[X[b] for b in _brng.choice(U, R_BASELINES, replace=False)] for _ in expl_users]
print(f'{len(expl_users)} explained users, {R_BASELINES} stochastic baselines each')

800 explained users, 5 stochastic baselines each


## 7. Full-data gradient (for the martingale diagnostic)

The exact gradient of the sampled-BCE objective costs two matmuls at this scale, so we can
evaluate it once per subepoch and compare it with what the stratum actually applied. The increment

$$ \delta_t \;=\; -\Delta\theta_t/\varepsilon \;-\; n_t\,\nabla L(\theta_t) $$

is the noise the scheme injects. Projected on a fixed random unit vector $v$, $M_n=\sum_{t\le n}
\langle \delta_t, v\rangle$ should look like a martingale (zero drift, $\mathrm{Var}\propto n$) if the
stratum gradients are conditionally unbiased, and should drift otherwise. Because $\theta$ moves
*within* a subepoch, this is an approximation at the subepoch scale — stated as such in the report.

In [14]:
def full_gradient(X, A, B, pos_u, pos_i, n_neg):
    """Exact gradient of the sampled-BCE objective (negatives in expectation)."""
    U, I = X.shape
    W = X @ A
    P = sigmoid(W @ B.T)
    m_u = np.bincount(pos_u, minlength=U).astype(np.float32)
    E = (n_neg / I) * m_u[:, None] * P
    np.add.at(E, (pos_u, pos_i), sigmoid((W[pos_u] * B[pos_i]).sum(1)) - 1.0)
    E /= len(pos_u)
    return X.T @ (E @ B), E.T @ W


def full_loss(X, A, B, pos_u, pos_i, n_neg):
    U, I = X.shape
    W = X @ A
    S = W @ B.T
    m_u = np.bincount(pos_u, minlength=U).astype(np.float32)
    neg = -(np.log(np.clip(1 - sigmoid(S), 1e-8, None)).sum(1) * m_u * n_neg / I).sum()
    posl = -np.log(np.clip(sigmoid(S[pos_u, pos_i]), 1e-8, None)).sum()
    return float((posl + neg) / len(pos_u))

## 8. The sweep

Every arm x seed, plus the centralized `SEQ` reference.

In [15]:

results = {'config': {**CFG, 'seeds': SEEDS, 'n_expl_users': int(N_EXPL_USERS),
                      'R_baselines': R_BASELINES, 'K_mask': K_MASK, 'U': int(U), 'I': int(I),
                      'train_nnz': int(X.sum()), 'fast': FAST},
           'runs': {}, 'partitions': {}, 'martingale': {}, 'ablation': {},
           'theorem2_control': {'max_abs_gap': float(np.max(gaps))},
           'theorem2_curve': t2curve}

for arm in ARMS:
    rl, cl, resh = make_partition(arm, X, G, year, age, CFG['d'], np.random.default_rng(0), item_nnz)
    bidx = block_index(pu, pi, rl, cl, CFG['d'])
    results['partitions'][arm] = {
        'row_sizes': np.bincount(rl, minlength=CFG['d']).tolist(),
        'col_sizes': np.bincount(cl, minlength=CFG['d']).tolist(),
        'block_nnz': [[len(bidx[(r, c)]) for c in range(CFG['d'])] for r in range(CFG['d'])],
        'reshuffle': bool(resh)}

models = {}
for seed in SEEDS:
    A, B, h = run_sequential(X, pu, pi, CFG, seed, evaluate)
    models[('SEQ', seed)] = (A, B); results['runs'][f'SEQ|{seed}'] = {'hist': h, 'diag': None}
    print('[%5.0fs] SEQ       seed=%d  val HR@10=%.4f' % (time.time()-t_start, seed, h[-1]['val_hr10']))
    for arm in ARMS:
        rl, cl, resh = make_partition(arm, X, G, year, age, CFG['d'],
                                      np.random.default_rng(seed), item_nnz)
        A, B, h, diag = run_dsgd(X, pu, pi, rl, cl, CFG, seed, evaluate, reshuffle=resh)
        models[(arm, seed)] = (A, B)
        results['runs'][f'{arm}|{seed}'] = {'hist': h, 'diag': {
            'overlap': float(np.mean(diag['overlap'])),
            'straggler': float(np.mean(diag['straggler'])),
            'gini': float(gini(np.array(diag['block_nnz']).ravel()))}}
        print('[%5.0fs] %-9s seed=%d  val HR@10=%.4f  contested=%.3f  straggler=%.2f'
              % (time.time()-t_start, arm, seed, h[-1]['val_hr10'],
                 np.mean(diag['overlap']), np.mean(diag['straggler'])))

[   48s] SEQ       seed=0  val HR@10=0.0890


[   76s] RAND      seed=0  val HR@10=0.0745  contested=0.679  straggler=1.11


[  105s] FIXED     seed=0  val HR@10=0.0830  contested=0.677  straggler=1.08


[  133s] GENRE     seed=0  val HR@10=0.0765  contested=0.659  straggler=1.42


[  162s] GENRE-BAL seed=0  val HR@10=0.0685  contested=0.667  straggler=1.08


...

[ 1075s] GENRE     seed=4  val HR@10=0.0705  contested=0.656  straggler=1.42


[ 1105s] GENRE-BAL seed=4  val HR@10=0.0695  contested=0.665  straggler=1.06


[ 1137s] YEAR      seed=4  val HR@10=0.0690  contested=0.665  straggler=1.12


[ 1168s] DEMOG     seed=4  val HR@10=0.0890  contested=0.637  straggler=1.58


[ 1198s] USER-HIST seed=4  val HR@10=0.0720  contested=0.587  straggler=1.25


## 9. Fidelity evaluation and paired statistics

In [16]:
def spinrec_fidelity(A, B, X, users, baselines_per_user, K=10, n_steps=20):
    """CDCG@k and POS@10 curves under progressive masking of top-attributed items."""
    S_full = (X[users] @ A) @ B.T
    j_star = S_full.argmax(1)
    Xu = X[users].copy()
    Xu[np.arange(len(users)), j_star] = 0.0

    attrs = []
    for n, u in enumerate(users):
        attrs.append(spinrec_attribution(A, B, Xu[n], j_star[n], baselines_per_user[n], n_steps))
    attrs = np.array(attrs)

    cdcg = np.zeros((len(users), K)); pos10 = np.zeros((len(users), K))
    order = [np.where(Xu[n] > 0)[0][np.argsort(-attrs[n][Xu[n] > 0])] for n in range(len(users))]
    for k in range(1, K + 1):
        Xm = Xu.copy()
        for n in range(len(users)):
            Xm[n, order[n][:k]] = 0.0
        S = (Xm @ A) @ B.T
        tgt = S[np.arange(len(users)), j_star]
        rank = (S > tgt[:, None]).sum(1) + 1
        cdcg[:, k - 1] = 1.0 / np.log2(rank + 1)
        pos10[:, k - 1] = (rank <= 10)
    return cdcg, pos10, attrs, j_star

In [17]:

fid = {}
for (arm, seed), (A, B) in models.items():
    cdcg, pos10, attrs, jstar = spinrec_fidelity(A, B, X, expl_users, baselines, K=K_MASK)
    fid[(arm, seed)] = dict(cdcg=cdcg, pos10=pos10, attrs=attrs)
    results['runs'][f'{arm}|{seed}']['fidelity'] = {
        'cdcg_mean': cdcg.mean(0).tolist(), 'pos10_mean': pos10.mean(0).tolist()}
    print('[%5.0fs] SPINRec %-9s seed=%d  CDCG@5=%.4f  POS@10 at k=5: %.3f'
          % (time.time()-t_start, arm, seed, cdcg.mean(0)[4], pos10.mean(0)[4]))

# paired per-user comparison against the paper's random blocking
paired = {}
for arm in ARMS:
    if arm == 'RAND':
        continue
    rows = []
    for seed in SEEDS:
        d5 = fid[(arm, seed)]['cdcg'][:, 4] - fid[('RAND', seed)]['cdcg'][:, 4]
        try:
            p = float(wilcoxon(d5).pvalue)
        except ValueError:
            p = 1.0
        boot = np.random.default_rng(1).choice(d5, (2000, len(d5))).mean(1)
        rows.append({'seed': seed, 'mean_diff': float(d5.mean()), 'p': p,
                     'ci': [float(np.quantile(boot, .025)), float(np.quantile(boot, .975))]})
    paired[arm] = rows
    print('%-10s  dCDCG@5 = %+.4f   worst-seed Wilcoxon p = %.3g'
          % (arm, np.mean([r['mean_diff'] for r in rows]), max(r['p'] for r in rows)))
results['paired_vs_rand'] = paired

# how much do the explanations themselves move relative to the centralized model?
stab = {}
for arm in ARMS + ['SEQ']:
    vals = []
    for seed in SEEDS:
        a1, a0 = fid[(arm, seed)]['attrs'], fid[('SEQ', seed)]['attrs']
        rs = [spearmanr(a1[n][X[expl_users[n]] > 0], a0[n][X[expl_users[n]] > 0]).statistic
              for n in range(len(expl_users))]
        vals.append(float(np.nanmean(rs)))
    stab[arm] = vals
results['attr_stability_vs_seq'] = stab
print('attribution Spearman vs SEQ:', {k: round(np.mean(v), 3) for k, v in stab.items()})

[ 1198s] SPINRec SEQ       seed=0  CDCG@5=0.3959  POS@10 at k=5: 0.636


[ 1199s] SPINRec RAND      seed=0  CDCG@5=0.4134  POS@10 at k=5: 0.614


[ 1199s] SPINRec FIXED     seed=0  CDCG@5=0.4289  POS@10 at k=5: 0.701


[ 1200s] SPINRec GENRE     seed=0  CDCG@5=0.3496  POS@10 at k=5: 0.521


[ 1200s] SPINRec GENRE-BAL seed=0  CDCG@5=0.3937  POS@10 at k=5: 0.551


...

[ 1218s] SPINRec YEAR      seed=4  CDCG@5=0.4249  POS@10 at k=5: 0.679


[ 1218s] SPINRec DEMOG     seed=4  CDCG@5=0.4379  POS@10 at k=5: 0.694


[ 1219s] SPINRec USER-HIST seed=4  CDCG@5=0.4163  POS@10 at k=5: 0.652
FIXED       dCDCG@5 = +0.0294   worst-seed Wilcoxon p = 0.0125
GENRE       dCDCG@5 = -0.0615   worst-seed Wilcoxon p = 0.0217
GENRE-BAL   dCDCG@5 = -0.0486   worst-seed Wilcoxon p = 0.117
YEAR        dCDCG@5 = -0.0345   worst-seed Wilcoxon p = 0.654
DEMOG       dCDCG@5 = -0.0407   worst-seed Wilcoxon p = 0.548


USER-HIST   dCDCG@5 = +0.0032   worst-seed Wilcoxon p = 0.00301


attribution Spearman vs SEQ: {'RAND': np.float64(0.283), 'FIXED': np.float64(0.363), 'GENRE': np.float64(0.374), 'GENRE-BAL': np.float64(0.368), 'YEAR': np.float64(0.336), 'DEMOG': np.float64(0.462), 'USER-HIST': np.float64(0.417), 'SEQ': np.float64(1.0)}


### Scaling in the number of blocks

The course's rates depend on the number of agents, so the natural question for our Theorem-2 gap is
how it scales with $d$. `DEMOG` is excluded because the ML-1M age brackets only define four groups.

In [18]:

results['d_sweep'] = {}
for d_ in D_GRID:
    cfg_d = {**CFG, 'd': d_}
    A_ck, B_ck = init_params(I, CFG['k'], 0)
    for _ in range(2):
        sgd_pass(X, A_ck, B_ck, pu, pi, np.arange(I), np.random.default_rng(77),
                 CFG['lr'], CFG['n_neg'], CFG['batch'])
    for arm in D_SWEEP_ARMS:
        rl, cl, _ = make_partition(arm, X, G, year, age, d_, np.random.default_rng(0), item_nnz)
        t2 = theorem2_curve(X, pu, pi, rl, cl, cfg_d, A_ck, B_ck, STEPS_GRID)
        hrs, ovs = [], []
        for seed in D_SEEDS:
            rl2, cl2, resh2 = make_partition(arm, X, G, year, age, d_,
                                             np.random.default_rng(seed), item_nnz)
            _, _, h, dg = run_dsgd(X, pu, pi, rl2, cl2, cfg_d, seed, evaluate, reshuffle=resh2)
            hrs.append(h[-1]['val_hr10']); ovs.append(float(np.mean(dg['overlap'])))
        results['d_sweep'][f'{arm}|{d_}'] = {'t2': t2, 'val_hr10': hrs, 'overlap': ovs}
        print('[%5.0fs] d=%d %-10s  Thm2(100 steps)=%.3f  val HR@10=%.4f  contested=%.3f'
              % (time.time()-t_start, d_, arm, t2[-1], np.mean(hrs), np.mean(ovs)))

[ 1312s] d=2 RAND        Thm2(100 steps)=0.078  val HR@10=0.0943  contested=0.429


[ 1405s] d=2 FIXED       Thm2(100 steps)=0.078  val HR@10=0.0920  contested=0.435


[ 1497s] d=2 GENRE       Thm2(100 steps)=0.087  val HR@10=0.0897  contested=0.415


[ 1580s] d=2 USER-HIST   Thm2(100 steps)=0.317  val HR@10=0.0913  contested=0.393


[ 1669s] d=4 RAND        Thm2(100 steps)=0.123  val HR@10=0.0737  contested=0.679


...

[ 1937s] d=4 USER-HIST   Thm2(100 steps)=0.323  val HR@10=0.0793  contested=0.595


[ 2030s] d=8 RAND        Thm2(100 steps)=0.132  val HR@10=0.0592  contested=0.800


[ 2123s] d=8 FIXED       Thm2(100 steps)=0.132  val HR@10=0.0712  contested=0.793


[ 2216s] d=8 GENRE       Thm2(100 steps)=0.125  val HR@10=0.0705  contested=0.769


[ 2304s] d=8 USER-HIST   Thm2(100 steps)=0.113  val HR@10=0.0755  contested=0.727


### What actually crosses the network

The course measures algorithms in bytes as well as epochs, so it is worth writing down what this
model costs. In classic MF a node ships its own row-block of $W$ and column-block of $H$, i.e.
$(m+n)k/d$ per node per subepoch. In the history-encoder model $\Delta A$ is dense over **all** item
rows -- every block touches the rows of $A$ for every item in its users' histories -- so a node ships
$(I + I/d)k$ instead. The ratio below is a property of the model, not of the partition.

In [19]:

k_, d_ = CFG['k'], CFG['d']
per_epoch_classic = d_ * d_ * ((U + I) / d_) * k_ * 4          # bytes: d subepochs x d nodes
per_epoch_hist    = d_ * d_ * (I + I / d_) * k_ * 4
results['communication'] = {
    'classic_mf_bytes_per_epoch': float(per_epoch_classic),
    'history_encoder_bytes_per_epoch': float(per_epoch_hist),
    'ratio': float(per_epoch_hist / per_epoch_classic),
    'dataset_bytes': float(int(X.sum()) * 8),
    'ratio_formula': '(d+1)*I / (U+I)'}
print('per epoch, d=%d:  classic MF %.2f MB   history-encoder %.2f MB   ratio %.2fx'
      % (d_, per_epoch_classic/1e6, per_epoch_hist/1e6, per_epoch_hist/per_epoch_classic))
print('training data itself: %.2f MB (sent once, or never in FL)' % (int(X.sum())*8/1e6))
print('ratio grows like (d+1)I/(U+I):', ['d=%d -> %.2fx' % (dd, (dd+1)*I/(U+I)) for dd in D_GRID])

per epoch, d=4:  classic MF 1.79 MB   history-encoder 3.84 MB   ratio 2.14x
training data itself: 1.35 MB (sent once, or never in FL)
ratio grows like (d+1)I/(U+I): ['d=2 -> 1.29x', 'd=4 -> 2.14x', 'd=8 -> 3.86x']


## 10. Martingale diagnostic and the negative-sampling ablation

In [20]:

for arm, weighted in [('RAND', False), ('FIXED', False), ('GENRE', False), ('GENRE', True)]:
    rl, cl, resh = make_partition(arm, X, G, year, age, CFG['d'], np.random.default_rng(0), item_nnz)
    _, _, _, diag = run_dsgd(X, pu, pi, rl, cl, CFG, 0, evaluate, reshuffle=resh,
                             martingale=True, weighted=weighted)
    key = f'{arm}{"-W" if weighted else ""}'
    results['martingale'][key] = {'inc': diag['mart_inc'], 'stratum_nnz': diag['stratum_nnz']}
    inc = np.array(diag['mart_inc']); M_n = np.cumsum(inc)
    print('%-9s  |M_n| = %.4g   max|increment| = %.4g   drift/sqrt(n) ratio = %.3f'
          % (key, abs(M_n[-1]), np.abs(inc).max(), abs(M_n[-1])/(np.abs(inc).max()*np.sqrt(len(inc)))))

# does the genre-shard hard-negative effect explain the accuracy gap?
for gneg in [False, True]:
    rl, cl, resh = make_partition('GENRE', X, G, year, age, CFG['d'],
                                  np.random.default_rng(0), item_nnz)
    A, B, h, _ = run_dsgd(X, pu, pi, rl, cl, CFG, 0, evaluate, reshuffle=resh,
                          global_negatives=gneg)
    cdcg, pos10, _, _ = spinrec_fidelity(A, B, X, expl_users, baselines, K=K_MASK)
    results['ablation'][f'GENRE_globalneg={gneg}'] = {
        'val_hr10': h[-1]['val_hr10'], 'cdcg_mean': cdcg.mean(0).tolist()}
    print('GENRE, global negatives=%-5s  val HR@10=%.4f  CDCG@5=%.4f'
          % (gneg, h[-1]['val_hr10'], cdcg.mean(0)[4]))

RAND       |M_n| = 11.13   max|increment| = 0.8884   drift/sqrt(n) ratio = 1.401


FIXED      |M_n| = 9.194   max|increment| = 0.798   drift/sqrt(n) ratio = 1.288


GENRE      |M_n| = 3.114   max|increment| = 0.6142   drift/sqrt(n) ratio = 0.567


GENRE-W    |M_n| = 2.772   max|increment| = 0.6022   drift/sqrt(n) ratio = 0.515


GENRE, global negatives=False  val HR@10=0.0765  CDCG@5=0.3496


GENRE, global negatives=True   val HR@10=0.0620  CDCG@5=0.9254


## 11. Figures and LaTeX tables for the report

In [21]:
R = results
CFG = results['config']   # the saved config, which also carries K_mask
d = CFG['d']
COL = {'SEQ': '#444444', 'RAND': '#1f77b4', 'FIXED': '#17becf', 'GENRE': '#d62728', 'GENRE-BAL': '#ff7f0e',
       'YEAR': '#9467bd', 'DEMOG': '#2ca02c', 'USER-HIST': '#8c564b', 'CLASSIC-MF': '#000000'}
plt.rcParams.update({'font.size': 13, 'figure.dpi': 150, 'savefig.bbox': 'tight',
                     'axes.grid': True, 'grid.alpha': .3, 'legend.frameon': False})


def save(fig, name):
    fig.savefig(FIG / f'{name}.png'); plt.close(fig); print('wrote', name)


def hist_of(arm, seed, key):
    return [h[key] for h in R['runs'][f'{arm}|{seed}']['hist']]


# --- fig1: block grids -------------------------------------------------------
show = ['RAND', 'GENRE', 'DEMOG', 'USER-HIST']
fig, axes = plt.subplots(2, 2, figsize=(7.4, 6.6), layout='constrained'); axes = axes.ravel()
for ax, arm in zip(axes, show):
    M = np.array(R['partitions'][arm]['block_nnz'])
    im = ax.imshow(M, cmap='viridis')
    for r in range(d):
        ax.add_patch(plt.Rectangle((r - .5, r - .5), 1, 1, fill=False, ec='w', lw=2))
    ax.set_title(f"{arm}\nmax/mean = {M.max()/M.mean():.2f}", fontsize=10.5)
    ax.set_xlabel('item group'); ax.set_xticks(range(d)); ax.set_yticks(range(d))
    ax.grid(False)
    if arm == show[0]:
        ax.set_ylabel('user group')
    fig.colorbar(im, ax=ax, fraction=.046)
fig.suptitle('Training pairs per block; white boxes = one stratum (a $d$-monomial)', fontsize=12)
save(fig, 'fig1_block_grid')

# --- fig2: load balance ------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(7.0, 7.2), layout='constrained')
for arm in ARMS:
    nnz = np.sort(np.array(R['partitions'][arm]['block_nnz']).ravel())[::-1]
    axes[0].plot(nnz / nnz.mean(), 'o-', ms=3, color=COL[arm], label=arm)
axes[0].axhline(1, color='k', lw=.8, ls=':')
axes[0].set_xlabel('block (sorted)'); axes[0].set_ylabel('nnz / mean nnz')
axes[0].set_title('Block load profile'); axes[0].legend(fontsize=9.5)

g = [np.mean([R['runs'][f'{a}|{s}']['diag']['gini'] for s in SEEDS]) for a in ARMS]
st = [np.mean([R['runs'][f'{a}|{s}']['diag']['straggler'] for s in SEEDS]) for a in ARMS]
x = np.arange(len(ARMS))
axes[1].bar(x - .2, g, .4, label='Gini(block nnz)', color='#4c72b0')
axes[1].bar(x + .2, np.array(st) - 1, .4, bottom=0, label='straggler ratio $-1$', color='#dd8452')
axes[1].set_xticks(x); axes[1].set_xticklabels(ARMS, rotation=30, ha='right')
axes[1].set_title('Imbalance and per-subepoch straggler cost'); axes[1].legend(fontsize=9.5)
save(fig, 'fig2_load_balance')

# --- fig3: convergence -------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(7.0, 7.2), layout='constrained')
for arm in ['SEQ'] + ARMS:
    for j, key in enumerate(['val_hr10', 'val_mpr']):
        Y = np.array([hist_of(arm, s, key) for s in SEEDS])
        ep = np.arange(1, Y.shape[1] + 1)
        axes[j].plot(ep, Y.mean(0), color=COL[arm], lw=1.6 if arm != 'SEQ' else 2,
                     ls='--' if arm == 'SEQ' else '-', label=arm)
        axes[j].fill_between(ep, Y.min(0), Y.max(0), color=COL[arm], alpha=.12)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('validation HR@10'); axes[0].legend(fontsize=9.5, ncol=2)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('validation MPR (lower = better)')
fig.suptitle('Recommendation quality per blocking scheme (band = min/max over seeds)', fontsize=12)
save(fig, 'fig3_convergence')

# --- fig4: Theorem 2 gap vs local steps -------------------------------------
tc = R['theorem2_curve']; steps = tc['steps']
fig, axes = plt.subplots(3, 1, figsize=(7.2, 9.0), layout='constrained')
for arm in ARMS + ['CLASSIC-MF']:
    y = np.maximum(np.array(tc[arm]), 1e-17)
    axes[0].plot(steps, y, 'o-', ms=3.5, color=COL[arm],
                 lw=2.2 if arm == 'CLASSIC-MF' else 1.3, label=arm)
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_ylim(1e-18, 3)
axes[0].set_ylabel(r'$\|\theta_{\rm par}-\theta_{\rm seq}\|\,/\,\|\theta_{\rm par}-\theta_0\|$')
axes[0].set_title('classic MF stays at machine zero', fontsize=11)
axes[0].legend(fontsize=9, loc='center left')
for arm in ARMS:
    axes[1].plot(steps, tc[arm], 'o-', ms=3.5, color=COL[arm], lw=1.3, label=arm)
axes[1].set_xscale('log')
axes[1].set_ylabel(r'$\|\theta_{\rm par}-\theta_{\rm seq}\|\,/\,\|\theta_{\rm par}-\theta_0\|$')
axes[1].set_title('history-encoder model, linear scale', fontsize=11)
axes[1].legend(fontsize=9, ncol=2)
for ax in axes:
    ax.set_xlabel('local mini-batch steps per block')
D_GRID = sorted({int(kk.split('|')[1]) for kk in R.get('d_sweep', {})})
D_ARMS = sorted({kk.split('|')[0] for kk in R.get('d_sweep', {})})
for arm in D_ARMS:
    ys = [R['d_sweep'][f'{arm}|{dd}']['t2'][-1] for dd in D_GRID]
    axes[2].plot(D_GRID, ys, 'o-', ms=5, color=COL[arm], lw=1.6, label=arm)
axes[2].set_xscale('log', base=2); axes[2].set_xticks(D_GRID)
axes[2].set_xticklabels([str(dd) for dd in D_GRID])
axes[2].set_xlabel('number of blocks per stratum, $d$')
axes[2].set_ylabel('Thm. 2 gap after 100 local steps')
axes[2].set_title('gap after 100 local steps versus the block count', fontsize=11)
axes[2].legend(fontsize=9, ncol=2)
fig.suptitle('Departure from Gemulla Thm. 2 within one subepoch', fontsize=12)
save(fig, 'fig4_theorem2_curve')

# --- fig9: d-sweep accuracy and contested mass (appendix) --------------------
fig, axes = plt.subplots(2, 1, figsize=(7.0, 7.0), layout='constrained')
for arm in D_ARMS:
    axes[0].plot(D_GRID, [np.mean(R['d_sweep'][f'{arm}|{dd}']['val_hr10']) for dd in D_GRID],
                 'o-', ms=5, color=COL[arm], lw=1.6, label=arm)
    axes[1].plot(D_GRID, [np.mean(R['d_sweep'][f'{arm}|{dd}']['overlap']) for dd in D_GRID],
                 'o-', ms=5, color=COL[arm], lw=1.6, label=arm)
seq = np.mean([hist_of('SEQ', s_, 'val_hr10')[-1] for s_ in SEEDS])
axes[0].axhline(seq, color=COL['SEQ'], ls='--', lw=1.5, label='SEQ (centralized)')
for ax, ylab in zip(axes, ['validation HR@10', 'contested gradient mass']):
    ax.set_xscale('log', base=2); ax.set_xticks(D_GRID)
    ax.set_xticklabels([str(dd) for dd in D_GRID])
    ax.set_xlabel('number of blocks per stratum, $d$'); ax.set_ylabel(ylab)
    ax.legend(fontsize=9, ncol=2)
fig.suptitle('Cost of parallelism: accuracy and contested mass versus $d$', fontsize=12)
save(fig, 'fig9_d_sweep')

# --- fig5/6: SPINRec fidelity ------------------------------------------------
K = CFG['K_mask']; ks = np.arange(1, K + 1)
for name, key, ylab in [('fig5_spinrec_cdcg', 'cdcg_mean', 'CDCG@k'),
                        ('fig6_spinrec_pos10', 'pos10_mean', 'POS@10')]:
    fig, ax = plt.subplots(figsize=(6.4, 4.2), layout='constrained')
    for arm in ['SEQ'] + ARMS:
        Y = np.array([R['runs'][f'{arm}|{s}']['fidelity'][key] for s in SEEDS])
        ax.plot(ks, Y.mean(0), 'o-', ms=3, color=COL[arm], ls='--' if arm == 'SEQ' else '-',
                label=arm)
        ax.fill_between(ks, Y.min(0), Y.max(0), color=COL[arm], alpha=.10)
    ax.set_xlabel('history items masked (k)'); ax.set_ylabel(ylab)
    ax.set_title(f'SPINRec deletion fidelity: {ylab}', fontsize=12); ax.legend(fontsize=9.5, ncol=2)
    save(fig, name)

# --- fig7: martingale / drift + Azuma ---------------------------------------
fig, ax = plt.subplots(figsize=(6.4, 4.2), layout='constrained')
for key, lab in [('RAND', 'RAND (random, re-permuted)'), ('FIXED', 'FIXED (random, fixed grid)'),
                 ('GENRE', 'GENRE (semantic item blocks)'),
                 ('GENRE-W', r'GENRE + SSGD weights $w_s\propto|Z_s|$')]:
    if key not in R['martingale']:
        continue
    inc = np.array(R['martingale'][key]['inc'])
    M = np.cumsum(inc); n = np.arange(1, len(M) + 1)
    ax.plot(n, M, label=lab, color='#7f0000' if key.endswith('-W') else COL.get(key, '#333'),
            ls='-.' if key.endswith('-W') else '-', lw=1.6)
    if key == 'RAND':
        c = np.abs(inc).max()
        env = c * np.sqrt(2 * n * np.log(2 / 0.05))
        ax.plot(n, env, 'k:', lw=1, label=r'Azuma envelope $\pm c\sqrt{2n\log(2/\delta)}$')
        ax.plot(n, -env, 'k:', lw=1)
ax.set_xlabel('subepoch $n$'); ax.set_ylabel(r'$\langle M_n, v\rangle$  (projected noise sum)')
ax.set_title('Is the stratum gradient noise a martingale difference?', fontsize=12)
ax.legend(fontsize=9, loc='lower left')
save(fig, 'fig7_martingale_azuma')

# --- fig8: dose-response (two candidate dose variables) ----------------------
def dcdcg(arm, s):
    return (np.array(R['runs'][f'{arm}|{s}']['fidelity']['cdcg_mean'])[4] -
            np.array(R['runs'][f'RAND|{s}']['fidelity']['cdcg_mean'])[4])


fig, axes = plt.subplots(2, 1, figsize=(7.0, 7.0), sharey=True, layout='constrained')
specs = [(lambda a, s_: R['runs'][f'{a}|{s_}']['diag']['overlap'],
          'contested gradient mass on shared $A$-rows'),
         (lambda a, s_: R['theorem2_curve'][a][-1],
          'Thm. 2 gap after 100 local steps')]
for ax, (xf, xlab) in zip(axes, specs):
    allx, ally = [], []
    for arm in ARMS:
        xs = [xf(arm, s_) for s_ in SEEDS]; ys = [dcdcg(arm, s_) for s_ in SEEDS]
        ax.scatter(xs, ys, color=COL[arm], s=42, zorder=3,
                   label=arm if ax is axes[0] else None)
        allx += xs; ally += ys
    b = np.polyfit(allx, ally, 1)
    xr = np.linspace(min(allx), max(allx), 10)
    r = np.corrcoef(allx, ally)[0, 1]
    ax.plot(xr, np.polyval(b, xr), 'k--', lw=1)
    ax.set_title(f'slope = {b[0]:+.2f},  $r$ = {r:+.2f}', fontsize=11)
    ax.axhline(0, color='k', lw=.8)
    ax.set_xlabel(xlab)
for ax in axes:
    ax.set_ylabel(r'$\Delta$ CDCG@5 vs RAND')
axes[0].legend(fontsize=9, ncol=2, loc='lower left')
fig.suptitle('Neither measure of the interchangeability violation predicts the fidelity shift',
             fontsize=12)
save(fig, 'fig8_fidelity_vs_overlap')
print('all figures written to', FIG)

wrote fig1_block_grid
wrote fig2_load_balance


wrote fig3_convergence


wrote fig4_theorem2_curve
wrote fig9_d_sweep
wrote fig5_spinrec_cdcg


/var/folders/nm/hggy1fjd227csp039zxfg5pm0000gn/T/ipykernel_11309/1189858413.py:125: UserWarning: linestyle is redundantly defined by the 'linestyle' keyword argument and the fmt string "o-" (-> linestyle='-'). The keyword argument will take precedence.
  ax.plot(ks, Y.mean(0), 'o-', ms=3, color=COL[arm], ls='--' if arm == 'SEQ' else '-',


wrote fig6_spinrec_pos10
wrote fig7_martingale_azuma


wrote fig8_fidelity_vs_overlap
all figures written to /Users/yagelalfasi/Documents/Agents/figures


In [22]:
# ============================ LaTeX tables ==================================
lines = []
lines.append(r'% --- Table 1: schemes and structural diagnostics')
lines.append(r'\begin{tabular}{lrrrrr}\hline')
lines.append(r'Scheme & Gini & straggler & contested & Thm.2 gap & val HR@10 \\')
lines.append(r'       &      & max/mean  & mass      & (100 steps) &         \\\hline')
for arm in ARMS:
    g = np.mean([R['runs'][f'{arm}|{s}']['diag']['gini'] for s in SEEDS])
    st = np.mean([R['runs'][f'{arm}|{s}']['diag']['straggler'] for s in SEEDS])
    ov = np.mean([R['runs'][f'{arm}|{s}']['diag']['overlap'] for s in SEEDS])
    hr = np.mean([hist_of(arm, s, 'val_hr10')[-1] for s in SEEDS])
    hrsd = np.std([hist_of(arm, s, 'val_hr10')[-1] for s in SEEDS])
    t2 = R['theorem2_curve'][arm][-1]
    lines.append(f'{arm} & {g:.3f} & {st:.2f} & {ov:.3f} & {t2:.3f} & {hr:.4f}$\\pm${hrsd:.4f} \\\\')
hr = np.mean([hist_of('SEQ', s, 'val_hr10')[-1] for s in SEEDS])
hrsd = np.std([hist_of('SEQ', s, 'val_hr10')[-1] for s in SEEDS])
lines.append(r'\hline')
lines.append(f'SEQ (centralized) & - & - & - & - & {hr:.4f}$\\pm${hrsd:.4f} \\\\')
lines.append(f"CLASSIC-MF control & - & - & - & {R['theorem2_curve']['CLASSIC-MF'][-1]:.1e} & -- \\\\")
lines.append(r'\hline\end{tabular}')

lines.append('')
lines.append(r'% --- Table 2: SPINRec fidelity, paired vs RAND')
from scipy import stats as _st
lines.append(r'\begin{tabular}{lrrrrr}\hline')
lines.append(r'Scheme & CDCG@5 & POS@10 at $k{=}5$ & $\Delta$CDCG@5 vs RAND & per-user $p$ & seed-level $p$ \\\hline')
for arm in ['SEQ'] + ARMS:
    c5 = np.mean([R['runs'][f'{arm}|{s}']['fidelity']['cdcg_mean'][4] for s in SEEDS])
    p5 = np.mean([R['runs'][f'{arm}|{s}']['fidelity']['pos10_mean'][4] for s in SEEDS])
    if arm in R.get('paired_vs_rand', {}):
        rows = R['paired_vs_rand'][arm]
        ds = np.array([r['mean_diff'] for r in rows])
        pv = max(r['p'] for r in rows)
        ps = float(_st.ttest_1samp(ds, 0).pvalue) if len(ds) > 1 else float('nan')
        extra = f'{ds.mean():+.4f} & $\\le${pv:.3f} & {ps:.3f}'
    else:
        extra = '- & - & -'
    lines.append(f'{arm} & {c5:.4f} & {p5:.3f} & {extra} \\\\')
lines.append(r'\hline\end{tabular}')

lines.append('')
lines.append(r'% --- Table 3: attribution stability vs centralized model (mean Spearman)')
lines.append(r'\begin{tabular}{lrr}\hline Scheme & Spearman $\rho$ vs SEQ attributions & seed-level $p$ vs RAND \\\hline')
_base = np.array(R['attr_stability_vs_seq']['RAND'])
for arm in ARMS:
    v = np.mean(R['attr_stability_vs_seq'][arm])
    sd = np.std(R['attr_stability_vs_seq'][arm])
    if arm == 'RAND':
        pp = '-'
    else:
        pp = f"{float(_st.ttest_1samp(np.array(R['attr_stability_vs_seq'][arm]) - _base, 0).pvalue):.3f}"
    lines.append(f'{arm} & {v:.3f}$\\pm${sd:.3f} & {pp} \\\\')
lines.append(r'\hline\end{tabular}')

lines.append('')
lines.append(r'% --- ablation + martingale summary')
for k, v in R['ablation'].items():
    lines.append(f"% {k}: val HR@10={v['val_hr10']:.4f}  CDCG@5={v['cdcg_mean'][4]:.4f}")
for k, v in R['martingale'].items():
    inc = np.array(v['inc']); M = np.cumsum(inc)
    lines.append(f"% martingale {k}: final |M_n|={abs(M[-1]):.4g}  mean inc={inc.mean():.4g}  "
                 f"max|inc|={np.abs(inc).max():.4g}  drift ratio={abs(M[-1])/(np.abs(inc).max()*np.sqrt(len(inc))):.3f}")
lines.append('')
lines.append(r'% --- Table 4: scaling in d')
lines.append(r'\begin{tabular}{lrrr}\hline')
lines.append(r'Scheme & $d$ & Thm.\,2 gap & val HR@10 \\\hline')
for _arm in sorted({kk.split('|')[0] for kk in R.get('d_sweep', {})}):
    for _dd in sorted({int(kk.split('|')[1]) for kk in R.get('d_sweep', {})}):
        _e = R['d_sweep'][f'{_arm}|{_dd}']
        lines.append(f"{_arm} & {_dd} & {_e['t2'][-1]:.3f} & {np.mean(_e['val_hr10']):.4f} \\\\")
lines.append(r'\hline\end{tabular}')
lines.append('')
_c = R.get('communication', {})
if _c:
    lines.append('%% communication per epoch: classic MF %.2f MB, history-encoder %.2f MB, ratio %.2fx (%s)'
                 % (_c['classic_mf_bytes_per_epoch']/1e6, _c['history_encoder_bytes_per_epoch']/1e6,
                    _c['ratio'], _c['ratio_formula']))
open(FIG / 'tables.tex', 'w').write('\n'.join(lines))
print('\n'.join(lines))

% --- Table 1: schemes and structural diagnostics
\begin{tabular}{lrrrrr}\hline
Scheme & Gini & straggler & contested & Thm.2 gap & val HR@10 \\
       &      & max/mean  & mass      & (100 steps) &         \\\hline
RAND & 0.053 & 1.11 & 0.678 & 0.123 & 0.0755$\pm$0.0086 \\
FIXED & 0.048 & 1.11 & 0.672 & 0.123 & 0.0793$\pm$0.0036 \\
GENRE & 0.155 & 1.42 & 0.658 & 0.143 & 0.0714$\pm$0.0031 \\
GENRE-BAL & 0.038 & 1.09 & 0.666 & 0.162 & 0.0699$\pm$0.0028 \\
YEAR & 0.074 & 1.15 & 0.663 & 0.151 & 0.0746$\pm$0.0041 \\
DEMOG & 0.173 & 1.58 & 0.639 & 0.127 & 0.0870$\pm$0.0059 \\
USER-HIST & 0.104 & 1.22 & 0.594 & 0.323 & 0.0775$\pm$0.0084 \\
\hline
SEQ (centralized) & -- & -- & -- & -- & 0.0880$\pm$0.0025 \\
CLASSIC-MF control & -- & -- & -- & 2.1e-18 & -- \\
\hline\end{tabular}

% --- Table 2: SPINRec fidelity, paired vs RAND
\begin{tabular}{lrrrrr}\hline
Scheme & CDCG@5 & POS@10 at $k{=}5$ & $\Delta$CDCG@5 vs RAND & per-user $p$ & seed-level $p$ \\\hline
SEQ & 0.3810 & 0.597 & -- & -- & -- \

In [23]:

with open(FIG / 'results.json', 'w') as f:
    json.dump(results, f, indent=1)
print('\nwrote results.json and 8 figures to', FIG)
print('total runtime: %.1f min' % ((time.time() - t_start) / 60))


wrote results.json and 8 figures to /Users/yagelalfasi/Documents/Agents/figures
total runtime: 41.4 min


## Notes and limitations

* This is a **single-process simulation** of DSGD: blocks are executed one after another but always
  from shared starting parameters, which reproduces the parallel arithmetic exactly. It therefore
  says nothing about real network or coordination cost — the paper's own bottleneck at scale.
* The straggler ratio is a *model* of makespan (subepoch cost $\propto$ largest block), not a timing
  measurement.
* The martingale increment is evaluated once per subepoch at that subepoch's starting parameters;
  parameter drift within a subepoch makes it an approximation.
* Accuracy differences between arms are partly driven by the negative-sampling protocol interacting
  with the item partition (see the ablation in Section 10) — fidelity numbers should be read together
  with the HR@10 column, not in isolation.